In [10]:
# ============================================================================
# 호영님 환경에 맞는 올바른 Import 코드
# ============================================================================

import sys
import os
from pathlib import Path

# ============================================================================
# 1. 프로젝트 루트 경로 자동 설정
# ============================================================================

def setup_investment_strategy_path():
    """investment_strategy 프로젝트 루트를 sys.path에 추가"""

    # 현재 노트북 위치
    current = Path.cwd()
    print(f"현재 위치: {current}")

    # investment_strategy 폴더 찾기 (상위로 올라가며)
    for parent in [current] + list(current.parents):
        if parent.name == 'investment_strategy':
            project_root = str(parent)
            if project_root not in sys.path:
                sys.path.insert(0, project_root)
            print(f"✓ 프로젝트 루트 추가: {project_root}")
            return project_root

    # 못 찾으면 직접 지정
    fallback = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy"
    if os.path.exists(fallback):
        if fallback not in sys.path:
            sys.path.insert(0, fallback)
        print(f"✓ Fallback 경로 사용: {fallback}")
        return fallback

    raise FileNotFoundError("investment_strategy 폴더를 찾을 수 없습니다.")

# 프로젝트 루트 설정
try:
    project_root = setup_investment_strategy_path()
except Exception as e:
    print(f"✗ 경로 설정 실패: {e}")
    # 수동으로 경로 지정
    project_root = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy"
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    print(f"수동 경로 설정: {project_root}")


# ============================================================================
# 2. 올바른 Import
# ============================================================================

# 주의: financial_data_integrator가 아니라 us_financial_data_integrator입니다!
from DATA.us_financial_data_integrator import integrate_financial_ratios
from DATA.stock_invest_function import get_db_host

# 현재 디렉토리의 모듈들
from financial_analysis_system import FinancialAnalysisSystem
from financial_forecast_extended import ForecastExtended

print("\n✓ 모든 모듈 import 완료!")
print("="*70)


# ============================================================================
# 3. 경로 확인 (디버깅용)
# ============================================================================

def check_module_locations():
    """모듈 위치 확인"""
    print("\n모듈 위치 확인:")
    print("="*70)

    modules_to_check = {
        'DATA.us_financial_data_integrator': None,
        'DATA.stock_invest_function': None,
        'financial_analysis_system': None,
        'financial_forecast_extended': None
    }

    for module_name in modules_to_check:
        try:
            if '.' in module_name:
                # 패키지 모듈
                parts = module_name.split('.')
                mod = __import__(module_name, fromlist=[parts[-1]])
            else:
                # 일반 모듈
                mod = __import__(module_name)

            if hasattr(mod, '__file__') and mod.__file__:
                print(f"✓ {module_name}")
                print(f"  → {mod.__file__}")
            else:
                print(f"✓ {module_name} (내장 모듈)")
        except ImportError as e:
            print(f"✗ {module_name}")
            print(f"  → Import 실패: {e}")

    print("="*70)

# 위치 확인 실행
check_module_locations()


# ============================================================================
# 4. sys.path 확인 (디버깅용)
# ============================================================================

def show_python_path():
    """현재 Python 경로 표시"""
    print("\nPython Path (상위 10개):")
    print("="*70)
    for i, path in enumerate(sys.path[:10], 1):
        print(f"{i}. {path}")
    if len(sys.path) > 10:
        print(f"... (외 {len(sys.path)-10}개)")
    print("="*70)

# 경로 확인 (필요시)
# show_python_path()

현재 위치: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\FS_Data_Analysis\fs_ratio_analysis
✓ 프로젝트 루트 추가: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy

✓ 모든 모듈 import 완료!

모듈 위치 확인:
✓ DATA.us_financial_data_integrator
  → C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\us_financial_data_integrator.py
✓ DATA.stock_invest_function
  → C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\stock_invest_function.py
✓ financial_analysis_system
  → C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\FS_Data_Analysis\fs_ratio_analysis\financial_analysis_system.py
✓ financial_forecast_extended
  → C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\FS_Data_Analysis\fs_ratio_analysis\financial_forecast_extended.py


In [11]:
# ============================================================================
# 2. DB 설정
# ============================================================================

db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}

print("✓ DB 설정 완료")
print(f"  Host: {db_info['host']}:{db_info['port']}\n")


# ============================================================================
# 3. 분석할 종목 설정
# ============================================================================

TICKER = "PLTR"
COMPANY_NAME = "Palantir Inc."

print(f"분석 대상: {COMPANY_NAME} ({TICKER})")
print("=" * 70 + "\n")


# ============================================================================
# 4. SEC 데이터 로드 및 재무비율 계산
# ============================================================================

print("[1/5] SEC 데이터 수집 중...")
headers = {"User-Agent": "Hoyoung Research <stox1224@gmail.com>"}

try:
    facts = fetch_company_facts(TICKER, headers=headers)
    parser = CompanyFactsParser(facts)
    normalizer = FinancialNormalizer(parser)
    df_normalized = normalizer.create_normalized_dataframe("quarterly")

    print(f"  ✓ SEC 데이터 로드 완료: {len(df_normalized)}개 분기")
    print(f"  기간: {df_normalized.index[0]} ~ {df_normalized.index[-1]}\n")
except Exception as e:
    print(f"  ✗ SEC 데이터 로드 실패: {e}")
    print("  CSV 파일에서 로드를 시도하거나 다른 티커를 사용하세요.\n")
    sys.exit(1)


print("[2/5] 재무비율 계산 중...")
df_with_ratios = integrate_financial_ratios(df_normalized)
print(f"  ✓ 재무비율 계산 완료: 총 {len(df_with_ratios.columns)}개 지표\n")


# ============================================================================
# 5. 분석 시스템 초기화
# ============================================================================

print("[3/5] 분석 시스템 초기화 중...")
analyzer = FinancialAnalysisSystem(df_with_ratios)

# 필요한 분석 미리 실행
analyzer.calculate_growth_rates()
analyzer.analyze_profitability_trends()
analyzer.analyze_financial_health()
analyzer.analyze_cash_flow()
analyzer.build_revenue_operating_income_model()

print("  ✓ 분석 시스템 초기화 완료\n")


# ============================================================================
# 6. DB 예측 데이터 확인 및 로드
# ============================================================================

print("[4/5] DB 예측 데이터 확인 중...")

# ForecastExtended 객체 생성
forecast = ForecastExtended(db_info, analyzer=analyzer)

# 사용 가능한 티커 목록 조회
print("\n[DB 정보 조회]")
available_tickers = forecast.get_available_tickers()
print(f"  사용 가능한 티커: {len(available_tickers)}개")

# 사용 가능한 지표 목록 조회
available_indicators = forecast.get_available_indicators()
print(f"  사용 가능한 지표: {available_indicators}")

# 현재 티커가 DB에 있는지 확인
if TICKER in available_tickers:
    print(f"  ✓ {TICKER}의 예측 데이터가 DB에 존재합니다!\n")
else:
    print(f"  ⚠ {TICKER}의 예측 데이터가 DB에 없습니다.")
    print(f"  회귀 모델만 사용하여 분석을 계속합니다.\n")


# ============================================================================
# 7-A. DB 예측 데이터가 있는 경우 (최신 기준일 자동 선택)
# ============================================================================

if TICKER in available_tickers:
    print("=== DB 예측 데이터 활용 모드 ===\n")

    # 옵션 1: 최신 예측 기준일 자동 선택 (forecast_date=None)
    print("  최신 예측 데이터 로드 중...")
    try:
        forecast.load_revenue_forecast(
            ticker=TICKER,
            indicator='prophet',  # 또는 'sarima', 'ets' 등
            forecast_date=None    # None = 최신 데이터
        )

        # 밸류에이션 예측도 로드 (있으면)
        forecast.load_valuation_forecast(
            ticker=TICKER,
            indicator='prophet_valuation',
            forecast_date=None
        )

        print("  ✓ DB 예측 데이터 로드 완료\n")

    except Exception as e:
        print(f"  ⚠ DB 예측 데이터 로드 실패: {e}")
        print("  회귀 모델만 사용합니다.\n")


# ============================================================================
# 7-B. 특정 예측 기준일을 지정하고 싶은 경우 (옵션)
# ============================================================================

# 특정 날짜를 지정하고 싶다면 아래 코드 사용
"""
if TICKER in available_tickers:
    print("=== 특정 예측 기준일 지정 모드 ===\n")

    # 특정 날짜 지정
    FORECAST_DATE = '2025-11-19'  # 원하는 날짜로 변경

    print(f"  예측 기준일: {FORECAST_DATE}")

    try:
        forecast.load_revenue_forecast(
            ticker=TICKER,
            indicator='prophet',
            forecast_date=FORECAST_DATE  # 특정 날짜 지정
        )

        forecast.load_valuation_forecast(
            ticker=TICKER,
            indicator='prophet_valuation',
            forecast_date=FORECAST_DATE
        )

        print("  ✓ DB 예측 데이터 로드 완료\n")

    except Exception as e:
        print(f"  ⚠ DB 예측 데이터 로드 실패: {e}")
        print("  회귀 모델만 사용합니다.\n")
"""


# ============================================================================
# 8. 확장 리포트 생성
# ============================================================================

print("[5/5] 확장 리포트 생성 중...")
print("=" * 70 + "\n")

try:
    forecast.generate_extended_report(
        ticker=TICKER,
        company_name=COMPANY_NAME,
        save_dir='./financial_reports',
        include_qoq=False,  # True로 변경하면 QoQ도 포함
        revenue_indicator='prophet',
        valuation_indicator='prophet_valuation'
    )

    print("\n" + "=" * 70)
    print("✓ 모든 분석 완료!")
    print("=" * 70)
    print(f"\n저장 위치: ./financial_reports/{TICKER}/")
    print("\n생성된 파일:")
    print("  - 10개 개별 차트 (PNG)")
    print("  - 1개 텍스트 리포트")
    print("\n차트 목록:")
    print(f"  {TICKER}_01_revenue_growth.png")
    print(f"  {TICKER}_02_operating_income_growth.png")
    print(f"  {TICKER}_03_net_income_growth.png")
    print(f"  {TICKER}_04_return_ratios.png")
    print(f"  {TICKER}_05_profit_margins.png")
    print(f"  {TICKER}_06_leverage.png")
    print(f"  {TICKER}_07_liquidity.png")
    print(f"  {TICKER}_08_cashflow.png")
    print(f"  {TICKER}_09_revenue_oi_model.png")
    print(f"  {TICKER}_10_forecast.png")
    print(f"  {TICKER}_extended_report.txt")

except Exception as e:
    print(f"\n✗ 리포트 생성 실패: {e}")
    import traceback
    traceback.print_exc()


# ============================================================================
# 9. 추가 분석 (옵션)
# ============================================================================

print("\n" + "=" * 70)
print("추가 분석 옵션")
print("=" * 70)

# 옵션 1: 차트만 생성 (리포트 없이)
print("\n[옵션 1] 차트만 생성하려면:")
print(f"  forecast.create_individual_charts(save_dir='./charts_{TICKER}', include_qoq=False)")

# 옵션 2: QoQ 포함 버전
print("\n[옵션 2] QoQ 성장률도 포함하려면:")
print(f"  forecast.create_individual_charts(save_dir='./charts_{TICKER}_qoq', include_qoq=True)")

# 옵션 3: 영업이익 예측만 확인
print("\n[옵션 3] 영업이익 예측만 확인하려면:")
print("  forecast_df = forecast.predict_operating_income()")
print("  print(forecast_df.head(10))")


# ============================================================================
# 10. 다른 티커 분석 (옵션)
# ============================================================================

print("\n" + "=" * 70)
print("다른 티커 분석")
print("=" * 70)

print("""
다른 티커를 분석하려면:

1. TICKER와 COMPANY_NAME 변경
   TICKER = "AAPL"
   COMPANY_NAME = "Apple Inc."

2. 이 스크립트 다시 실행

또는 함수로 만들어서 여러 종목 분석:

def analyze_company(ticker, company_name):
    # 위의 코드들을 함수 안에 넣기
    ...

# 여러 종목 분석
companies = [
    ("AAPL", "Apple Inc."),
    ("MSFT", "Microsoft Corp."),
    ("GOOGL", "Alphabet Inc.")
]

for ticker, name in companies:
    analyze_company(ticker, name)
""")


# ============================================================================
# 끝
# ============================================================================

print("\n" + "=" * 70)
print("실행 완료!")
print("=" * 70)

✓ DB 설정 완료
  Host: 192.168.0.230:3307

분석 대상: Palantir Inc. (PLTR)

[1/5] SEC 데이터 수집 중...
  ✓ SEC 데이터 로드 완료: 28개 분기
  기간: 2017-12-31 00:00:00 ~ 2025-09-30 00:00:00

[2/5] 재무비율 계산 중...
재무비율 계산 중...
  - 수익성 비율 계산...
  - 레버리지 비율 계산...
  - 유동성 비율 계산...
  - 효율성 비율 계산...
재무비율 계산 완료!
  ✓ 재무비율 계산 완료: 총 43개 지표

[3/5] 분석 시스템 초기화 중...

[1/8] 성장률 분석 중...
[2/8] 수익성 추세 분석 중...
[3/8] 재무건전성 분석 중...
[5/8] 현금흐름 분석 중...
[7/8] 매출-영업이익 예측 모델 구축 중...
  회귀식: Operating Income = 1.0180 * Revenue + -2396245627.47
  R² = 0.7732
  평균 영업이익률 = -28.46%
  ✓ 분석 시스템 초기화 완료

[4/5] DB 예측 데이터 확인 중...
✓ DB 연결 성공: 192.168.0.230:3307

[DB 정보 조회]
  사용 가능한 티커: 845개
  사용 가능한 지표: ['revenue_billions_esq_forecast', 'revenue_billions_lstm_forecast', 'revenue_billions_prophet_forecast', 'revenue_billions_sarima_noexog']
  ✓ PLTR의 예측 데이터가 DB에 존재합니다!

=== DB 예측 데이터 활용 모드 ===

  최신 예측 데이터 로드 중...
✗ 예측 기준일 조회 실패: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MariaD

No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


  [8/10] Cash Flow...
  [9/10] Revenue-OI Model...
  [10/10] Revenue & Operating Income Forecast...

✓ 모든 차트 생성 완료! (총 10개)
  저장 위치: ./financial_reports\PLTR

[3/3] 텍스트 리포트 생성...
  리포트 저장: ./financial_reports\PLTR\PLTR_extended_report.txt

✓ Extended Report 생성 완료!
  저장 위치: ./financial_reports\PLTR
  차트: 10개 (개별 PNG)
  QoQ 성장률: 미포함 (YoY만)

✓ 모든 분석 완료!

저장 위치: ./financial_reports/PLTR/

생성된 파일:
  - 10개 개별 차트 (PNG)
  - 1개 텍스트 리포트

차트 목록:
  PLTR_01_revenue_growth.png
  PLTR_02_operating_income_growth.png
  PLTR_03_net_income_growth.png
  PLTR_04_return_ratios.png
  PLTR_05_profit_margins.png
  PLTR_06_leverage.png
  PLTR_07_liquidity.png
  PLTR_08_cashflow.png
  PLTR_09_revenue_oi_model.png
  PLTR_10_forecast.png
  PLTR_extended_report.txt

추가 분석 옵션

[옵션 1] 차트만 생성하려면:
  forecast.create_individual_charts(save_dir='./charts_PLTR', include_qoq=False)

[옵션 2] QoQ 성장률도 포함하려면:
  forecast.create_individual_charts(save_dir='./charts_PLTR_qoq', include_qoq=True)

[옵션 3] 영업이익 예측만 확인하려면:
  forecast_d